In [3]:
import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

c:\Users\Kayan\OneDrive\Desktop\SE Factory\week1_assigment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
file_path = "ds_salaries.csv"

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "ruchi798/data-science-job-salaries",
  file_path,
)
df.to_csv("data/ds_salaries.csv", index=False)
df.head()

,Unnamed: 0,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,0,2020,MI,FT,Data Scientist,70000,EUR,79833,DE,0,DE,L
1,1,2020,SE,FT,Machine Learning Scientist,260000,USD,260000,JP,0,JP,S
2,2,2020,SE,FT,Big Data Engineer,85000,GBP,109024,GB,50,GB,M
3,3,2020,MI,FT,Product Data Analyst,20000,USD,20000,HN,0,HN,S
4,4,2020,SE,FT,Machine Learning Engineer,150000,USD,150000,US,50,US,L


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 607 entries, 0 to 606
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Unnamed: 0          607 non-null    int64
 1   work_year           607 non-null    int64
 2   experience_level    607 non-null    str  
 3   employment_type     607 non-null    str  
 4   job_title           607 non-null    str  
 5   salary              607 non-null    int64
 6   salary_currency     607 non-null    str  
 7   salary_in_usd       607 non-null    int64
 8   employee_residence  607 non-null    str  
 9   remote_ratio        607 non-null    int64
 10  company_location    607 non-null    str  
 11  company_size        607 non-null    str  
dtypes: int64(5), str(7)
memory usage: 57.0 KB


In [6]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,607.0,303.000000,1.753701e+02,0.0,151.5,303.0,454.5,606.0
work_year,607.0,2021.405272,6.921330e-01,2020.0,2021.0,2022.0,2022.0,2022.0
salary,607.0,324000.062603,1.544357e+06,4000.0,70000.0,115000.0,165000.0,30400000.0
salary_in_usd,607.0,112297.869852,7.095726e+04,2859.0,62726.0,101570.0,150000.0,600000.0
remote_ratio,607.0,70.922570,4.070913e+01,0.0,50.0,100.0,100.0,100.0


In [7]:
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values)

Missing values per column:
Unnamed: 0            0
work_year             0
experience_level      0
employment_type       0
job_title             0
salary                0
salary_currency       0
salary_in_usd         0
employee_residence    0
remote_ratio          0
company_location      0
company_size          0
dtype: int64


In [8]:
df['job_title'].value_counts()

job_title
Data Scientist                              143
Data Engineer                               132
Data Analyst                                 97
Machine Learning Engineer                    41
Research Scientist                           16
Data Science Manager                         12
Data Architect                               11
Machine Learning Scientist                    8
Big Data Engineer                             8
Data Science Consultant                       7
Director of Data Science                      7
AI Scientist                                  7
Principal Data Scientist                      7
Data Analytics Manager                        7
Lead Data Engineer                            6
BI Data Analyst                               6
ML Engineer                                   6
Computer Vision Engineer                      6
Business Data Analyst                         5
Data Engineering Manager                      5
Head of Data                  

In [9]:
top_titles = df["job_title"].value_counts().nlargest(25).index

df["job_title_clean"] = df["job_title"].where(
    df["job_title"].isin(top_titles), "Other"
)
df["job_title_clean"].value_counts()

job_title_clean
Data Scientist                        143
Data Engineer                         132
Data Analyst                           97
Other                                  48
Machine Learning Engineer              41
Research Scientist                     16
Data Science Manager                   12
Data Architect                         11
Machine Learning Scientist              8
Big Data Engineer                       8
Data Science Consultant                 7
Director of Data Science                7
AI Scientist                            7
Principal Data Scientist                7
Data Analytics Manager                  7
Lead Data Engineer                      6
BI Data Analyst                         6
ML Engineer                             6
Computer Vision Engineer                6
Business Data Analyst                   5
Data Engineering Manager                5
Head of Data                            5
Applied Data Scientist                  5
Data Analytics Eng

In [10]:
for col in df[['work_year','remote_ratio','experience_level', 'employment_type',
       'job_title_clean', 'employee_residence',
       'company_location', 'company_size']].columns:
    unique_vals = df[col].dropna().unique()
    print(f"{col}: {len(unique_vals)} unique values")
    print(unique_vals)
    print("-" * 50)

work_year: 3 unique values
[2020 2021 2022]
--------------------------------------------------
remote_ratio: 3 unique values
[  0  50 100]
--------------------------------------------------
experience_level: 4 unique values
<StringArray>
['MI', 'SE', 'EN', 'EX']
Length: 4, dtype: str
--------------------------------------------------
employment_type: 4 unique values
<StringArray>
['FT', 'CT', 'PT', 'FL']
Length: 4, dtype: str
--------------------------------------------------
job_title_clean: 26 unique values
<StringArray>
[                    'Data Scientist',         'Machine Learning Scientist',
                  'Big Data Engineer',                              'Other',
          'Machine Learning Engineer',                       'Data Analyst',
              'Business Data Analyst',                 'Lead Data Engineer',
                      'Data Engineer',            'Data Science Consultant',
                    'BI Data Analyst',           'Director of Data Science',
         

In [16]:
X_categorical = df[['experience_level', 'employment_type',
       'job_title_clean', 'employee_residence',
       'company_location', 'company_size']]
X_numerical = df[['work_year','remote_ratio']]

feature_names = X_categorical.columns

encoder = OneHotEncoder(sparse_output=False)
X_categorical_encoded = encoder.fit_transform(X_categorical)

X_categorical_encoded = pd.DataFrame(X_categorical_encoded,
                                     columns=encoder.get_feature_names_out(feature_names),
                                     index=df.index)

X = pd.concat([X_numerical, X_categorical_encoded], axis= 1)
y = df['salary_in_usd']



In [17]:
print(X.shape)
print(y.shape)

(607, 146)
(607,)


In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(485, 146)
(122, 146)
(485,)
(122,)


In [19]:
model = DecisionTreeRegressor(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error:", mae)
print("R-squared:", r2)
# Mean Absolute Error: 31048.802343565403
# R-squared: 0.4629243531786478


Mean Absolute Error: 28987.724474712944
R-squared: 0.561188730844526


In [20]:
joblib.dump({
    "model": model,
    "encoder": encoder,
    "feature_names": X.columns.tolist()
}, "models/salary_model_bundle.pkl")

['models/salary_model_bundle.pkl']